# TESSERA — Fetching Pre-Computed Embeddings

This notebook uses `geotessera` to fetch real TESSERA embeddings for a geographic region of interest and inspects the returned xarray Dataset.

`geotessera.fetch()` queries the Cambridge TESSERA embedding service, which has global coverage at ~10 m spatial resolution aligned to the Sentinel-2 grid. Embeddings are 128-dimensional and encode the multi-temporal Sentinel-1/2 signature of each pixel across the full annual cycle.

## References
- GeoTessera docs: https://github.com/ucam-eo/geotessera
- TESSERA GitHub: https://github.com/ucam-eo/tessera

## 1. Setup

In [ ]:
import geotessera
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box

print(f"geotessera version: {geotessera.__version__}")

## 2. Define a Region of Interest

In [ ]:
# Small area around Cambridge, UK (Sentinel-2 tile 30UXC)
# Use a small bounding box to keep download size manageable
bbox = {
    "min_lon": 0.05,
    "min_lat": 52.15,
    "max_lon": 0.20,
    "max_lat": 52.25,
}

print("Region of interest:")
for k, v in bbox.items():
    print(f"  {k}: {v}")

roi_geom = box(bbox["min_lon"], bbox["min_lat"], bbox["max_lon"], bbox["max_lat"])
roi_gdf = gpd.GeoDataFrame(geometry=[roi_geom], crs="EPSG:4326")

ax = roi_gdf.plot(figsize=(4, 4), edgecolor="red", facecolor="none")
ax.set_title("Region of Interest")
plt.tight_layout()
plt.show()

## 3. Fetch TESSERA Embeddings

In [ ]:
# Note: geotessera.fetch() requires internet access to the Cambridge TESSERA API
# The dataset may require registration — see https://github.com/ucam-eo/geotessera

try:
    ds = geotessera.fetch(
        min_lon=bbox["min_lon"],
        min_lat=bbox["min_lat"],
        max_lon=bbox["max_lon"],
        max_lat=bbox["max_lat"],
        year=2023,  # embedding year
    )
    print("Embeddings fetched successfully.")
    print(ds)

except Exception as e:
    print(f"API not available: {e}")
    print("\nFalling back to simulated data for demonstration purposes.")

    # Simulate the structure of a real TESSERA response
    n_lat, n_lon, embed_dim = 100, 150, 128
    lat = np.linspace(bbox["min_lat"], bbox["max_lat"], n_lat)
    lon = np.linspace(bbox["min_lon"], bbox["max_lon"], n_lon)

    # Simulate two land cover regions: ~60% farmland, ~40% urban
    data = np.random.randn(n_lat, n_lon, embed_dim).astype(np.float32)
    data[60:, :, :] += 2.0  # urban area has a different embedding distribution

    ds = xr.Dataset(
        {"embedding": (["lat", "lon", "feature"], data)},
        coords={"lat": lat, "lon": lon, "feature": np.arange(embed_dim)},
    )
    print("\nSimulated dataset created.")
    print(ds)

## 4. Inspect the Embedding Dataset

In [ ]:
emb = ds["embedding"].values  # shape: [lat, lon, feature]

print(f"Embedding array shape: {emb.shape}")
print(f"  {emb.shape[0]} latitude pixels")
print(f"  {emb.shape[1]} longitude pixels")
print(f"  {emb.shape[2]} embedding features")
print(f"\nValue range: [{emb.min():.3f}, {emb.max():.3f}]")
print(f"Memory footprint: {emb.nbytes / 1e6:.1f} MB")

## 5. Visualize Embeddings as False-Color RGB

In [ ]:
from sklearn.decomposition import PCA

# Reduce 128 embedding dims to 3 for RGB visualization
n_lat, n_lon, n_feat = emb.shape
emb_flat = emb.reshape(-1, n_feat)

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(emb_flat).reshape(n_lat, n_lon, 3)

# Scale to [0, 1]
emb_rgb = (emb_pca - emb_pca.min()) / (emb_pca.max() - emb_pca.min())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(emb_rgb, extent=[bbox["min_lon"], bbox["max_lon"], bbox["min_lat"], bbox["max_lat"]])
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title("TESSERA embeddings — PCA RGB (dims 1-3)")

variance = pca.explained_variance_ratio_
axes[1].bar(["PC1", "PC2", "PC3"], variance * 100, color=["red", "green", "blue"])
axes[1].set_ylabel("Variance explained (%)")
axes[1].set_title(f"Top-3 PCA components ({sum(variance)*100:.1f}% total variance)")

plt.tight_layout()
plt.show()

## 6. Save Embeddings Locally for Downstream Use

In [ ]:
import zarr

ds.to_zarr("tessera_embeddings.zarr", mode="w")
print("Saved embeddings to tessera_embeddings.zarr")
print("\nReload with: xr.open_zarr('tessera_embeddings.zarr')")
print("\nContinue to 2-downstream_classification.ipynb to train a classifier on these embeddings.")